<a href="https://colab.research.google.com/github/venkata18167/CSA6301---THREAT-INTELLIGENCE-AND-NETWORK-SECURITY/blob/main/30_Stateful_Firewall_Connection_Tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
def handle_outbound_syn(packet, state_table, allow_rules):
    """
    packet: {"src","sport","dst","dport"} outbound SYN.
    If (dst, dport) is permitted by allow_rules,
    record the connection in the state table.
    """

    key = (packet["dst"], packet["dport"])

    if key in allow_rules:
        state_table[
            (
                packet["src"],
                packet["sport"],
                packet["dst"],
                packet["dport"]
            )
        ] = "ESTABLISHED"

        return "allow"

    return "deny"
def handle_inbound_packet(packet, state_table):
    """
    packet: {"src","sport","dst","dport"} inbound packet.
    Allow only if it matches an existing outbound connection.
    """
    key = (
        packet["dst"],
        packet["dport"],
        packet["src"],
        packet["sport"]
    )

    if key in state_table:
        return "allow"

    return "deny"

def test_experiment2():

    state_table = {}
    allow_rules = {
        ("93.184.216.34", 443)
    }
    outbound = {
        "src": "10.0.0.20",
        "sport": 51000,
        "dst": "93.184.216.34",
        "dport": 443
    }

    result1 = handle_outbound_syn(outbound, state_table, allow_rules)
    print("Outbound SYN:", result1)

    assert result1 == "allow"

    assert (
        outbound["src"],
        outbound["sport"],
        outbound["dst"],
        outbound["dport"]
    ) in state_table

    response = {
        "src": "93.184.216.34",
        "sport": 443,
        "dst": "10.0.0.20",
        "dport": 51000
    }
    result2 = handle_inbound_packet(response, state_table)
    print("Legitimate Response:", result2)

    assert result2 == "allow"

    unsolicited = {
        "src": "203.0.113.9",
        "sport": 4444,
        "dst": "10.0.0.20",
        "dport": 51000
    }
    result3 = handle_inbound_packet(unsolicited, state_table)
    print("Unsolicited Packet:", result3)
    assert result3 == "deny"
    print("\nState Table:")
    print(state_table)
    print("\nAll test cases passed.")
test_experiment2()

Outbound SYN: allow
Legitimate Response: allow
Unsolicited Packet: deny

State Table:
{('10.0.0.20', 51000, '93.184.216.34', 443): 'ESTABLISHED'}

All test cases passed.
